## 9. Conclusiones

### Tendencias (2000-2015)
- La esperanza de vida y el PIB aumentaron en los 6 paises durante el periodo.
- **Zimbabwe** tuvo la mejora mas dramatica en esperanza de vida (+14.7 anos), probablemente por avances en tratamiento de VIH/SIDA.
- **China** lidera en crecimiento de PIB (~900%), reflejando su rapida industrializacion.

### Relacion PIB - Esperanza de Vida
- Existe una correlacion **positiva pero no lineal**: mayor PIB se asocia con mayor esperanza de vida, pero la relacion se aplana en niveles altos de PIB.
- La regresion lineal simple tiene un ajuste limitado, lo que sugiere que factores como el sistema de salud, la educacion y las politicas publicas son igualmente importantes.

### Pruebas estadisticas
- La esperanza de vida de Chile es significativamente superior al promedio global (p < 0.001).
- Existe una diferencia significativa entre paises de alto y bajo PIB (p < 0.001), aunque Zimbabwe como outlier influye considerablemente en este resultado.

### Implicaciones
- El PIB por si solo no determina la esperanza de vida. Paises con PIB moderado (como Chile) pueden lograr alta esperanza de vida con buenas politicas de salud publica.
- Para predicciones mas precisas, seria necesario un modelo multivariable que incluya gasto en salud, educacion y desigualdad (indice de Gini).

---
*Analisis comparativo de datos del Banco Mundial y la OMS, 6 paises, periodo 2000-2015.*

In [ ]:
# Clasificar paises por PIB
high_gdp_countries = ['United States of America', 'China', 'Germany']
df['gdp_group'] = df['Country'].apply(lambda x: 'Alto PIB' if x in high_gdp_countries else 'Bajo PIB')

high = df[df['gdp_group'] == 'Alto PIB']['life_expectancy']
low = df[df['gdp_group'] == 'Bajo PIB']['life_expectancy']

t_stat2, p_val2 = stats.ttest_ind(high, low)

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(high, bins=10, alpha=0.6, color='#3498db', label=f'Alto PIB (media: {high.mean():.1f})', density=True)
ax.hist(low, bins=10, alpha=0.6, color='#e74c3c', label=f'Bajo PIB (media: {low.mean():.1f})', density=True)
ax.axvline(high.mean(), color='#3498db', linestyle='--', linewidth=2)
ax.axvline(low.mean(), color='#e74c3c', linestyle='--', linewidth=2)
ax.set_title('Distribucion de Esperanza de Vida: Alto vs Bajo PIB', fontsize=14, fontweight='bold')
ax.set_xlabel('Esperanza de Vida (anos)', fontsize=12)
ax.set_ylabel('Densidad', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"t-statistic: {t_stat2:.4f}")
print(f"p-value: {p_val2:.6f}")
print(f"\nResultado: {'Rechazamos H0' if p_val2 < 0.05 else 'No rechazamos H0'} (alpha = 0.05)")
print("La diferencia ES significativa, pero Zimbabwe influye fuertemente en el grupo de bajo PIB.")

### 8.2 Paises de alto PIB vs bajo PIB: hay diferencia en esperanza de vida?

Clasificamos los paises en dos grupos segun su PIB y evaluamos si existe una diferencia significativa.

- **H0:** No hay diferencia en esperanza de vida entre paises de alto y bajo PIB
- **H1:** Existe una diferencia significativa

In [ ]:
# T-test: Chile vs media global
mean_chile = df[df['Country'] == 'Chile']['life_expectancy'].mean()
mean_global = df['life_expectancy'].mean()

t_stat, p_val = stats.ttest_1samp(df[df['Country'] == 'Chile']['life_expectancy'], mean_global)

print(f"Media Chile: {mean_chile:.2f} anos")
print(f"Media Global: {mean_global:.2f} anos")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_val:.8f}")
print(f"\nResultado: {'Rechazamos H0' if p_val < 0.05 else 'No rechazamos H0'} (alpha = 0.05)")
print(f"La esperanza de vida de Chile es significativamente {'mayor' if mean_chile > mean_global else 'menor'} que el promedio global.")

## 8. Pruebas de Hipotesis

### 8.1 Es la esperanza de vida de Chile diferente del promedio global?

- **H0:** La media de esperanza de vida de Chile es igual a la media global
- **H1:** La media de esperanza de vida de Chile es diferente de la media global

In [ ]:
# Regresion lineal simple
from numpy.polynomial import polynomial as P

slope, intercept, r_value, p_value, std_err = stats.linregress(df['gdp_billions'], df['life_expectancy'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter con linea de regresion
x_range = np.linspace(df['gdp_billions'].min(), df['gdp_billions'].max(), 100)
y_pred = intercept + slope * x_range

for country, color in zip(countries, colors):
    data = df[df['Country'] == country]
    axes[0].scatter(data['gdp_billions'], data['life_expectancy'], color=color, s=50, alpha=0.7, label=country)
axes[0].plot(x_range, y_pred, color='red', linewidth=2, linestyle='--', label=f'Regresion (R²={r_value**2:.3f})')
axes[0].set_title('Regresion Lineal: PIB vs Esperanza de Vida', fontsize=13, fontweight='bold')
axes[0].set_xlabel('PIB (miles de millones USD)', fontsize=11)
axes[0].set_ylabel('Esperanza de Vida (anos)', fontsize=11)
axes[0].legend(fontsize=9)

# Residuos
y_fitted = intercept + slope * df['gdp_billions']
residuals = df['life_expectancy'] - y_fitted
axes[1].scatter(y_fitted, residuals, alpha=0.5, color='#3498db', s=40)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Analisis de Residuos', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Valores Predichos', fontsize=11)
axes[1].set_ylabel('Residuos', fontsize=11)

plt.tight_layout()
plt.show()

print(f"Pendiente: {slope:.4f} anos por cada mil millones USD de PIB")
print(f"Intercepto: {intercept:.2f} anos")
print(f"R²: {r_value**2:.4f}")
print(f"P-value: {p_value:.6f}")
print(f"\nEl modelo explica solo el {r_value**2*100:.1f}% de la variacion.")
print("Los residuos muestran patrones claros, indicando que la relacion NO es lineal.")

## 7. Regresion Lineal: PIB como Predictor de Esperanza de Vida

Probamos si un modelo lineal simple puede capturar la relacion entre PIB y esperanza de vida.

In [ ]:
# Scatter plot global: PIB vs Esperanza de Vida
fig, ax = plt.subplots(figsize=(14, 8))

for country, color in zip(countries, colors):
    data = df[df['Country'] == country]
    ax.scatter(data['gdp_billions'], data['life_expectancy'], label=country, 
              color=color, s=80, alpha=0.8, edgecolors='white')

ax.set_title('PIB vs Esperanza de Vida (todos los paises)', fontsize=15, fontweight='bold')
ax.set_xlabel('PIB (miles de millones USD)', fontsize=12)
ax.set_ylabel('Esperanza de Vida (anos)', fontsize=12)
ax.legend(title='Pais', fontsize=11, title_fontsize=12)
plt.tight_layout()
plt.show()

# Correlacion de Pearson por pais
print("=" * 50)
print("CORRELACION DE PEARSON POR PAIS")
print("=" * 50)
for country in countries:
    data = df[df['Country'] == country]
    corr, pval = stats.pearsonr(data['gdp_billions'], data['life_expectancy'])
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"
    print(f"  {country:<30} r = {corr:.3f}  (p = {pval:.4f}) {sig}")

corr_global, pval_global = stats.pearsonr(df['gdp_billions'], df['life_expectancy'])
print(f"\n  {'GLOBAL':<30} r = {corr_global:.3f}  (p = {pval_global:.4f})")

## 6. Correlacion entre PIB y Esperanza de Vida

Esta es la pregunta central del estudio: existe una relacion entre la riqueza de un pais y la longevidad de sus ciudadanos?

In [ ]:
# Histogramas de PIB por pais
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (country, color) in enumerate(zip(countries, colors)):
    data = df[df['Country'] == country]['gdp_billions']
    axes[i].hist(data, bins=10, color=color, edgecolor='white', alpha=0.85)
    axes[i].axvline(data.mean(), color='black', linestyle='--', linewidth=1.5, label=f'Media: ${data.mean():.0f}B')
    axes[i].set_title(country, fontsize=13, fontweight='bold')
    axes[i].set_xlabel('PIB (miles de millones USD)', fontsize=10)
    axes[i].set_ylabel('Frecuencia', fontsize=10)
    axes[i].legend(fontsize=9)

plt.suptitle('Distribucion del PIB por Pais', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("China muestra una distribucion fuertemente sesgada a la derecha,")
print("reflejando su crecimiento exponencial durante el periodo.")

In [ ]:
# Distribucion de esperanza de vida por pais (boxplot + violin)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sns.boxplot(data=df, x='Country', y='life_expectancy', hue='Country', palette=colors, ax=axes[0], legend=False)
axes[0].set_title('Boxplot: Esperanza de Vida por Pais', fontsize=13, fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Esperanza de Vida (anos)', fontsize=11)
axes[0].tick_params(axis='x', rotation=25)

sns.violinplot(data=df, x='Country', y='life_expectancy', hue='Country', palette=colors, ax=axes[1], legend=False)
axes[1].set_title('Violin Plot: Distribucion de Esperanza de Vida', fontsize=13, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Esperanza de Vida (anos)', fontsize=11)
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()

print("Zimbabwe muestra la mayor variabilidad (rango de ~44 a 59 anos),")
print("lo que refleja los cambios dramaticos en su sistema de salud durante este periodo.")
print("Los paises desarrollados (Alemania, EE.UU.) muestran distribuciones mas compactas.")

## 5. Distribucion de Variables

Analizamos la forma de las distribuciones de esperanza de vida y PIB para cada pais, evaluando si son normales o presentan sesgo.

In [ ]:
# Tasa de crecimiento del PIB y esperanza de vida (2000 vs 2015)
growth_data = []
for country in countries:
    c_data = df[df['Country'] == country]
    gdp_init = c_data[c_data['Year'] == 2000]['gdp_billions'].values[0]
    gdp_final = c_data[c_data['Year'] == 2015]['gdp_billions'].values[0]
    le_init = c_data[c_data['Year'] == 2000]['life_expectancy'].values[0]
    le_final = c_data[c_data['Year'] == 2015]['life_expectancy'].values[0]
    growth_data.append({
        'Country': country,
        'GDP Growth (%)': ((gdp_final - gdp_init) / gdp_init) * 100,
        'LE Change (years)': le_final - le_init
    })

growth_df = pd.DataFrame(growth_data)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Crecimiento PIB
bars1 = axes[0].barh(growth_df['Country'], growth_df['GDP Growth (%)'], color=colors, edgecolor='white')
axes[0].set_title('Crecimiento del PIB 2000-2015 (%)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Crecimiento (%)', fontsize=11)
for bar, val in zip(bars1, growth_df['GDP Growth (%)']):
    axes[0].text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, f'{val:.0f}%', va='center', fontsize=10)

# Cambio esperanza de vida
bars2 = axes[1].barh(growth_df['Country'], growth_df['LE Change (years)'], color=colors, edgecolor='white')
axes[1].set_title('Cambio en Esperanza de Vida 2000-2015 (anos)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Cambio (anos)', fontsize=11)
for bar, val in zip(bars2, growth_df['LE Change (years)']):
    axes[1].text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2, f'+{val:.1f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print("China lidera en crecimiento de PIB, pero Zimbabwe lidera en mejora de esperanza de vida.")
print("Esto sugiere que la relacion PIB-esperanza de vida no es lineal ni directa.")

In [ ]:
# Tendencia del PIB por pais
fig, ax = plt.subplots(figsize=(14, 7))

for country, color in zip(countries, colors):
    data = df[df['Country'] == country]
    ax.plot(data['Year'], data['gdp_billions'], marker='o', label=country, 
            color=color, linewidth=2.5, markersize=6)

ax.set_title('Evolucion del PIB por Pais (2000-2015)', fontsize=15, fontweight='bold')
ax.set_xlabel('Ano', fontsize=12)
ax.set_ylabel('PIB (miles de millones USD)', fontsize=12)
ax.legend(title='Pais', fontsize=11, title_fontsize=12)
ax.set_xticks(range(2000, 2016))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print("Estados Unidos domina en PIB absoluto, seguido por China con un crecimiento exponencial.")
print("Chile y Zimbabwe tienen PIB similar en escala, pero trayectorias muy diferentes.")

## 4. Evolucion del PIB por Pais

In [ ]:
# Esperanza de vida individual por pais (subplots)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (country, color) in enumerate(zip(countries, colors)):
    data = df[df['Country'] == country]
    axes[i].plot(data['Year'], data['life_expectancy'], marker='o', color=color, linewidth=2.5)
    axes[i].fill_between(data['Year'], data['life_expectancy'], alpha=0.15, color=color)
    axes[i].set_title(country, fontsize=13, fontweight='bold')
    axes[i].set_ylabel('Esperanza de Vida', fontsize=10)
    axes[i].tick_params(axis='x', rotation=45)
    
    # Mostrar cambio total
    change = data['life_expectancy'].iloc[-1] - data['life_expectancy'].iloc[0]
    axes[i].text(0.05, 0.95, f'+{change:.1f} anos', transform=axes[i].transAxes,
                fontsize=12, fontweight='bold', va='top', color=color)

plt.suptitle('Esperanza de Vida por Pais (2000-2015)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Tendencia de esperanza de vida por pais
fig, ax = plt.subplots(figsize=(14, 7))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']
countries = df['Country'].unique()

for country, color in zip(countries, colors):
    data = df[df['Country'] == country]
    ax.plot(data['Year'], data['life_expectancy'], marker='o', label=country, 
            color=color, linewidth=2.5, markersize=6)

ax.set_title('Evolucion de la Esperanza de Vida por Pais (2000-2015)', fontsize=15, fontweight='bold')
ax.set_xlabel('Ano', fontsize=12)
ax.set_ylabel('Esperanza de Vida (anos)', fontsize=12)
ax.legend(title='Pais', fontsize=11, title_fontsize=12)
ax.set_xticks(range(2000, 2016))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print("Todos los paises muestran una tendencia ascendente.")
print("Zimbabwe presenta el crecimiento mas dramatico (+14.7 anos), probablemente")
print("debido a mejoras en el tratamiento del VIH/SIDA a partir de 2005.")

## 3. Evolucion de la Esperanza de Vida por Pais

Analizamos como ha cambiado la esperanza de vida en cada pais durante el periodo 2000-2015.

In [ ]:
# Estadisticas descriptivas por pais
summary = df.groupby('Country').agg(
    life_exp_mean=('life_expectancy', 'mean'),
    life_exp_std=('life_expectancy', 'std'),
    gdp_mean_B=('gdp_billions', 'mean'),
    gdp_std_B=('gdp_billions', 'std')
).round(2)

print("=" * 70)
print("ESTADISTICAS DESCRIPTIVAS POR PAIS")
print("=" * 70)
print(summary.to_string())
print()
print(f"Esperanza de vida global promedio: {df['life_expectancy'].mean():.1f} anos")
print(f"PIB global promedio: ${df['gdp_billions'].mean():.1f} mil millones")

## 2. Exploracion Inicial

In [ ]:
df = pd.read_csv('all_data.csv')

# Renombrar columnas para facilitar el analisis
df = df.rename(columns={
    'Life expectancy at birth (years)': 'life_expectancy',
    'GDP': 'gdp'
})

# Convertir PIB a miles de millones para mejor legibilidad
df['gdp_billions'] = df['gdp'] / 1e9

print(f"Dataset: {df.shape[0]} registros, {df.shape[1]} variables")
print(f"Paises: {df['Country'].nunique()} -> {list(df['Country'].unique())}")
print(f"Periodo: {df['Year'].min()} - {df['Year'].max()}")
print()
df.head(10)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12

print("Librerias cargadas correctamente")

## 1. Importacion de Librerias y Datos

# Analisis de Esperanza de Vida y PIB: Estudio Comparativo entre Paises (2000-2015)

Este proyecto investiga la relacion entre el producto interno bruto (PIB) y la esperanza de vida en seis paises representativos de diferentes niveles de desarrollo economico: **Chile, China, Alemania, Mexico, Estados Unidos y Zimbabwe**.

**Preguntas de investigacion:**
1. Como ha evolucionado la esperanza de vida y el PIB en cada pais?
2. Existe una correlacion significativa entre PIB y esperanza de vida?
3. Que distribucion siguen estas variables por pais?
4. Un modelo de regresion lineal puede predecir la esperanza de vida a partir del PIB?
5. Existen diferencias significativas entre paises de alto y bajo PIB?

**Dataset:** Datos del Banco Mundial y la OMS, 6 paises, 16 anos (2000-2015).